# 4. CNN

Module importing

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

Library importing

In [2]:
import numpy as np
import torch
from src.cnn import train_cnn, predict_cnn
from src.evaluate import score

CNN training

In [3]:
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

train_ids = list(tables["train"].index)
rng = np.random.default_rng(0)
val_ids = list(rng.choice(train_ids, size=8, replace=False))
fit_ids = [c for c in train_ids if c not in val_ids]

curves = lambda ids: [cells[i]["qdlin"] for i in ids]
lives = lambda ids: tables["train"].loc[ids, "cycle_life"].values

cnns = [train_cnn(curves(fit_ids), lives(fit_ids), curves(val_ids), lives(val_ids), seed=s)
        for s in range(5)]

for split in ["primary_test", "secondary_test"]:
    ids, truth = list(tables[split].index), tables[split]["cycle_life"]
    per_seed = np.array([predict_cnn(m, curves(ids)) for m in cnns])
    singles = [score(truth, p)["RMSE (cycles)"] for p in per_seed]
    print(split, "single seeds:", singles, "| ensemble:", score(truth, per_seed.mean(0)))

results["1D CNN (5-seed ensemble)"] = {
    s: (t["cycle_life"], np.mean([predict_cnn(m, curves(list(t.index))) for m in cnns], axis=0))
    for s, t in tables.items()
}

device: cuda


NameError: name 'tables' is not defined